# Overfitting and Underfitting

### Underfitting (error due to bias)

- Means that the model is too simplistic. Poor fit between the model and the data because we have oversimplified the problem
- Sometimes referred to as error due to bias, the training data may be biased and this bias may be incorporated into the model in a way that oversimplifies it.

### Overfitting (error due to variance)

- Means that the model is too complicated. Fit between the model and the training data is too specific, it will perform very well on the training data but will fail to generalize to new data.
- Sometimes referred to as error due to variance, there are random irrelevant differences among the data points in the training data and the model have fit so closely to these irrelevant differences that it performs poorly when trying to use it with the testing data.

### Applying This to Neural Networks
Generally speaking, underfitting tends to happen with neural networks that have overly simple architecture, while overfitting tends to happen with models that are highly complex.

# Model Complexity Graph

![Model Complexity Graph](imgs/model-complexity-graph.png)

If we moniotor the training error and the testing error, the train error will go down and at some point the testing error will start to increase. This indicaes overfiting and it should have stopped prior to that.

# Regularization

![Activation Functions](imgs/activation-functions.png)

When applying the sigmoid (δ) to small values such as x1 + x2, we have a nice slope, but when we multiply the linear function by 10, do the sigmoid then the predictions get much better since they are closer to 0 and 1 (but the function becomes much steeper and it's harder to do the gradient descent).

So the right one is bad for gradient descent and the incorrectly classified items will generate large errors thus hard to tune the model to correct them.

## How do we prevent this type of overfitting from happening?

The problem is that large coefficients are leading to overfitting, so we need to adjust the error function by penalizing large weights.

We have the function:

$$
-\frac{1}{m} \sum_{i=1}^{m} \left[ (1 - y_i)\ln(1 - \hat{y}_i) + y_i \ln(\hat{y}_i) \right]
$$

| Symbol | Meaning |
|--------|---------|
| $m$ | Number of training examples |
| $y_i$ | True label for example $i$ (0 or 1) |
| $\hat{y}_i$ | Predicted probability for example $i$ |
| $\sum_{i=1}^{m}$ | Sum over all training examples |
| $-\frac{1}{m}$ | Averages the total error and flips the sign, since log-probabilities are negative |

### How it works

This is the same binary cross-entropy error we've used before, it heavily penalizes confident-but-wrong predictions. Right now, though, it only looks at how well the predictions match the labels; it has no idea how large or small the weights are. That's exactly the gap we need to close: a model can drive this error very low by using large weights to make extremely sharp, overconfident predictions, which is what leads to overfitting. To fix that, we'll need to add a *penalty term* based on the weights themselves, so the error function also "cares" about keeping weights small.

> **Real-life example:** Imagine grading an exam purely on "did you get the right answer," with no penalty for using extreme, convoluted, all-or-nothing reasoning to get there. A student could memorize edge cases and answer with absolute certainty every time — great grade, but no real understanding. Penalizing large weights is like also grading *how the student got there*, discouraging answers that swing to extremes just to squeeze out a lower score on this equation.

We want to take the function above and add a term that is big when the weights are big. There are two common ways to do this:

### L1 Regularization

$$
+\ \lambda \left( |w_1| + \dots + |w_n| \right)
$$

Summing the absolute values of the weights, times a constant $\lambda$.

### L2 Regularization

$$
+\ \lambda \left( w_1^2 + \dots + w_n^2 \right)
$$

Summing the squares of the weights, times the same constant $\lambda$.

| Symbol | Meaning |
|--------|---------|
| $w_1, \dots, w_n$ | The model's weights |
| $\lambda$ | Regularization parameter — controls how much we penalize large weights |
| $\vert w_i \vert$ | Absolute value of weight $i$ (used in L1) |
| $w_i^2$ | Square of weight $i$ (used in L2) |

### How it works

Both terms get added on top of the error function from before, so the total error now grows whenever weights get large — not just when predictions are wrong. This gives the model an incentive to keep weights small unless a large weight is *really* earning its keep by reducing prediction error. $\lambda$ controls the trade-off: a small $\lambda$ barely penalizes big weights (closer to the original, overfitting-prone error), while a large $\lambda$ pushes hard for small weights, risking underfitting if pushed too far.

L1 and L2 differ in behavior: L1 (absolute values) tends to push some weights all the way to zero, effectively performing feature selection by ignoring less useful inputs entirely. L2 (squares) tends to shrink all weights smoothly toward zero without eliminating any completely, so it spreads the "blame" more evenly across weights.

### L1 Regularization

- L1 tends to result in sparse vectors. That means small weights will tend to go to zero.
- If we want to reduce the number of weights and end up with a small set, we can use L1.
- L1 is also good for feature selection. Sometimes we have a problem with hundreds of features, and L1 regularization will help us select which ones are important, turning the rest into zeroes.

### L2 Regularization

- L2 tends not to favor sparse vectors since it tries to maintain all the weights homogeneously small.
- L2 gives better results for training models so it's the one we'll use the most.

> **Real-life example:** Think of $\lambda$ as a household budget limit. L2 is like trimming a little from every expense category to stay under budget — everything gets a bit smaller, but nothing is cut entirely. L1 is like deciding to cancel a few subscriptions outright while leaving the rest unchanged — some "weights" (expenses) go to exactly zero, others stay as they were.

# Dropout

When training NN, sometimes one part of the network has very large weights and it ends up dominating the training, while another part of the network doesn't really play much of a role (so it doesn't get trained).

To solve this we can use `dropout`, where we "turn off" part of the network and let the rest work.

- We go through the epochs and randomly turn off some of the nodes. This forces the other nodes to pick up the slack and take a larger part in the training;
- To drop nodes we give the algorithm a parameter that indicates the probability that each node will get dropped during each epoch. E.g.: if we set this parameter to 0.2, this means that during each epoch, each node has a 20% probability of being turned off.
- Some nodes may get turned off more than others and some may never get turned off, this is OK since doing it over and over on avg each node will get approximately the same treatment.

# Local Minima and Random Restart

Gradient descent goes toward the direction it can most descrease the error and then takes a step in that direction. However, if there are multiple low points in the solution space, gradient descent may end up leading us to a `local minima` (solution where the error is at the lowest point in the local area but not the lowest point overwall).

A way to solve the problem is to use `random restart`. We start and restart from a few different random places and go gradient descend from all of them. This increases the probability that we'll get to the global minimum, or at least a good local minimum.

# Other Activation Functions

We've been focused on the sigmoid function, but it can lead to some problems. Let's check out some of the issues that come up with the sigmoid function, and then check other activation functions.

### The Vanish Gradient Problem

The problem is that the sigmoid curve gets pretty flat on the sides, so if we calculate the derivative at a point far to the right or far to the left, this derivative is almost zero. This is problematic because it is the derivative that tells us what direction to move in. This is especially problematic in most linear perceptrons.

### Other functions

#### Hyperbolic Tangent

$$
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$

| Symbol | Meaning |
|--------|---------|
| $x$ | The input value |
| $e^x$, $e^{-x}$ | Exponential of $x$ and of $-x$ |
| $\tanh(x)$ | Output, ranging between $-1$ and $1$ |

**How it works:** Similar in shape to the sigmoid, but since the output range is $(-1, 1)$ instead of $(0, 1)$, the curve is steeper around the middle — which means its derivatives are larger there. This small difference has a big impact on neural networks, since larger derivatives mean stronger gradient signals during training, making it less prone to the vanishing gradient problem than sigmoid.

#### ReLU (Rectified Linear Unit)

$$
\text{relu}(x) =
\begin{cases}
x & \text{if } x \geq 0 \\
0 & \text{if } x < 0
\end{cases}
$$

| Symbol | Meaning |
|--------|---------|
| $x$ | The input value |
| $\text{relu}(x)$ | Output — same as input if positive, otherwise 0 |

**How it works:** If the input is positive, ReLU just returns the same value unchanged; if it's negative, it returns zero. This function is used a lot instead of sigmoid or tanh, and it can improve training significantly without sacrificing much accuracy — because its derivative is a constant $1$ for any positive input, so the gradient doesn't shrink no matter how large the input gets (unlike sigmoid/tanh, which flatten out and shrink the gradient at the extremes).

> **Real-life example:** Think of a bouncer at a club door. ReLU is a bouncer who lets everyone with a valid ID (positive input) through exactly as they are, no questions asked, but turns away anyone without one (negative input) completely — zero entry. Compare that to sigmoid/tanh, which act more like a strict dress code that "squashes" everyone's outfit judgment into a narrow range, becoming less and less sensitive to how flashy or plain an outfit is once it's already clearly "very fancy" or "very plain."


# Batch vs. Stochastic Gradient Descent

## Batch Gradient Descent

In order to decrease error, we take a bunch of steps following the negative of the gradient of the error function. Each step is called an **epoch**, and within a single epoch:

1. We take our input (**all** of our data) and run it through the entire network.
2. We compute our predictions and calculate the error (how far the predictions are from the actual labels).
3. We back-propagate this error to update the weights in the network, giving us a better decision boundary for our data.

**The problem:** if we have a large number of data points, this process involves huge matrix computations on every single epoch, which uses a lot of memory and time.

## Stochastic Gradient Descent

If the data is well-distributed, a *subset* of the data can already give us enough information about the gradient — we don't need to look at everything at once. The process instead becomes:

1. Split the data into several batches.
2. Take the points in the first batch and run them through the network.
3. Calculate the error and its gradient for that batch only.
4. Back-propagate to update the weights (defining a slightly better decision boundary).
5. Repeat the steps above for each remaining batch.

With this approach we take *multiple* smaller steps instead of one large one. Each individual step is less accurate, since it's based on only a slice of the data — but in practice, taking many slightly-inaccurate steps gets us to a good solution faster than waiting to take one perfectly-accurate one.

| | Batch Gradient Descent | Stochastic Gradient Descent |
|---|---|---|
| Data used per step | All of the data | One batch (subset) of the data |
| Steps per epoch | 1 | Several (one per batch) |
| Accuracy per step | High | Lower |
| Memory / compute cost | High | Lower |

> **Real-life example:** Imagine trying to find the best route to work by studying a complete, perfectly detailed city map before taking a single step (batch) — accurate, but slow to even get started. Stochastic gradient descent is more like checking the map block by block as you walk, adjusting your direction slightly after each block. You might zig-zag a little more than the "perfect route" would, but you're moving the whole time, and you'll typically arrive just as well — often faster overall.

# Learning Rate Decay

| Learning rate | Effect |
|---|---|
| Large | Takes large steps, which can make training faster — but the large steps may cause it to miss (overshoot) the minimum |
| Small | Takes small steps, which can make training slower — but it makes steady progress and has a better chance of arriving at the local minimum |

If the model is not working, a good general rule of thumb is to try decreasing the learning rate. The best learning rates are those that decrease as the algorithm gets closer to a solution.

# Momentum

This is also a technique to solve the `local minimum` problem. Momentum is a constant $\beta$ between 0 and 1.

We use $\beta$ to get a sort of weighted average of the previous steps:

$$
\text{step}(n) + \beta \cdot \text{step}(n-1) + \beta^2 \cdot \text{step}(n-2) + \beta^3 \cdot \text{step}(n-3) + \dots
$$

| Symbol | Meaning |
|--------|---------|
| $\text{step}(n)$ | The most recent step (gets full weight, i.e. multiplied by 1) |
| $\text{step}(n-1)$ | The step before that (multiplied by $\beta$) |
| $\text{step}(n-2)$ | Two steps back (multiplied by $\beta^2$) |
| $\beta$ | Momentum constant, between 0 and 1 |

### How it works

The most recent step is multiplied by 1, the one before it by $\beta$, the one before that by $\beta^2$, the one before that by $\beta^3$, and so on. Since $\beta$ is between 0 and 1, raising it to increasingly large powers makes the value smaller and smaller. This means steps that happened a long time ago get multiplied by tiny values and end up mattering much less than the ones that happened recently — a "weighted memory" of past steps rather than an equal average of all of them.

This momentum can carry us over small "humps" in the error surface and help us escape shallow local minima to discover better ones. Once we reach the global minimum, momentum will still be pushing us slightly past it, but by then its effect is small enough that we settle back down near the correct point.

> **Real-life example:** Think of rolling a ball down a bumpy hill. A ball with no momentum stops the instant it hits a small dip (a local minimum), even if a much deeper valley is just beyond the next bump. A ball that's already rolling (with momentum) carries enough speed from its recent motion to roll straight through small dips and bumps, guided mostly by where it's been moving lately rather than getting stuck at the very next obstacle.